In [1]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from torchvision import datasets
from torchvision.datasets import CIFAR10
import torchvision.transforms as transforms
from torchvision.transforms import ToTensor
from torch.utils.data import Subset
from torch.utils.data import DataLoader
import torch.nn as nn
from torch import optim
from opacus import PrivacyEngine
from torch.autograd import Variable
import numpy as np
from torch.nn.utils import clip_grad_norm_
import model_base
from model_base import BaseTrainer
import compute_privacy_sgm
from compute_privacy_sgm import *
import mirror_descent
from mirror_descent import Trainer
import math
import copy

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2023, 0.1994, 0.2010])
])

In [4]:
train_data = CIFAR10(
    root = 'data',
    train = True,
    transform = transform,
    download = True,
)
train_public = Subset(train_data, list(range(2000)))
train_private = Subset(train_data, list(range(2000, len(train_data))))
test_data = CIFAR10(
    root = 'data',
    train = False,
    transform = transform
)

In [5]:
loaders = {
    'train_public' : torch.utils.data.DataLoader(train_public,
                                          batch_size=100,
                                          shuffle=True,),
    'train_private' : torch.utils.data.DataLoader(train_private,
                                          batch_size=100,
                                          shuffle=True,),
    'test'  : torch.utils.data.DataLoader(test_data,
                                          batch_size=100,
                                          shuffle=True,),
}

In [6]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.out = nn.Linear(64 * 4 * 4, 10)
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = x.view(x.size(0), -1)
        output = self.out(x)
        return output, x

In [7]:
cnn = CNN()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn = cnn.to(device)

In [15]:
params = {
    'model': cnn,
    'public_loader': loaders['train_public'],
    'train_loader': loaders['train_private'],
    'test_loader': loaders['test'],
    'epochs': 25,
    'batch_size': 100,
    'num_microbatches': 100,
    'lr': 0.01,
    'sigma': 1.0,
    'clipping_bound': 1.0,
    'eval_every_epoch': 1,
    'device': device,
    'public_bs': 100,
}

In [16]:
cnntrain = Trainer(params)

In [17]:
cnntrain.train()

epoch 0, test accuracy 0.10170
DP-SGD with sampling rate = 0.208% and noise_multiplier = 1.0 iterated over 0 steps satisfies differential privacy with eps = 0 and delta = 4.166666666666667e-06.
The optimal RDP order is 1.1.
The privacy estimate is likely to be improved by expanding the set of orders.

Epoch 0, Batch 0/480, Current Loss: 2.3022


KeyboardInterrupt: 

In [ ]:
#testing not set up